In [ ]:
import time

start_time = time.time()
print("Installing a special library for Notebook 01...")

!pip install uv
!uv pip install geopandas pyarrow tqdm requests

elapsed_time = time.time() - start_time
print(f"\n✓ Instalasi selesai! (Waktu: {elapsed_time:.1f} detik)")

In [ ]:
import requests
import geopandas as gpd
import os
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE DATABASE SNOWFLAKE_LEARNING_DB").collect()
session.sql("USE SCHEMA PUBLIC").collect()

data_stage_path = "@SNOWFLAKE_LEARNING_DB.PUBLIC.EXTERNAL_DATA_STAGE"
print("Snowflake session and Module ready!")

In [ ]:
data_to_download = {
    "zaf_pop_2025_CN_100m_R2025A_v1.tif": "https://data.worldpop.org/GIS/Population/Global_2015_2030/R2025A/2025/ZAF/v1/100m/constrained/zaf_pop_2025_CN_100m_R2025A_v1.tif",
    "BasinATLAS_v1.gdb.zip": "https://figshare.com/ndownloader/files/20082137",
    "RiverATLAS_v1.gdb.zip": "https://figshare.com/ndownloader/files/20087321"
}

for file_name, url in data_to_download.items():
    local_path = f"/tmp/{file_name}"

    if os.path.exists(local_path):
        print(f"Finding file remains {file_name} previously. Delete to re-download cleanly....")
        os.remove(local_path)
        
    print(f" Initiating a connection to {file_name} ...")
    response = requests.get(url, stream=True, timeout=900)
    response.raise_for_status()
    
    total_size_in_bytes = int(response.headers.get('content-length', 0))
    block_size = 1024 * 1024 # 1 Megabyte chunk
    

    progress_bar = tqdm(total=total_size_in_bytes, unit='iB', unit_scale=True, desc=file_name)
    
    with open(local_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=block_size): 
            if chunk: 
                f.write(chunk)
                progress_bar.update(len(chunk)) 
                
    progress_bar.close()
    

    if total_size_in_bytes != 0 and progress_bar.n != total_size_in_bytes:
        print(f"ERROR: An error occurred, {file_name} did not download completely.")
    else:
        print(f" {file_name} successfully downloaded in full!\n")

In [ ]:
print("Begin Convertion GeoDatabase (.gdb) to GeoParquet (.parquet)...")

# Konversi BasinATLAS
if os.path.exists("/tmp/BasinATLAS_v1.gdb.zip"):
    print("Reading BasinATLAS GDB to RAM...")
    basin_gdf = gpd.read_file("zip:///tmp/BasinATLAS_v1.gdb.zip")
    print("Converting BasinATLAS to Parquet...")
    basin_gdf.to_parquet("/tmp/BasinATLAS_v1.parquet", compression='snappy')
    print("✓ BasinATLAS Parquet complete!")
else:
    print("File ZIP BasinATLAS tidak ditemukan!")

# Konversi RiverATLAS
if os.path.exists("/tmp/RiverATLAS_v1.gdb.zip"):
    print("\nReading RiverATLAS GDB to RAM...")
    river_gdf = gpd.read_file("zip:///tmp/RiverATLAS_v1.gdb.zip")
    print("Converting RiverATLAS to Parquet...")
    river_gdf.to_parquet("/tmp/RiverATLAS_v1.parquet", compression='snappy')
    print("✓ RiverATLAS Parquet complete!")
else:
    print("File ZIP RiverATLAS tidak ditemukan!")
     

In [ ]:
print("Uploading file statis final ke Snowflake Stage...")

files_to_upload = [
    "/tmp/zaf_pop_2025_CN_100m_R2025A_v1.tif",
    "/tmp/BasinATLAS_v1.parquet", 
    "/tmp/RiverATLAS_v1.parquet"  
]

for file_path in files_to_upload:
    if os.path.exists(file_path):
        file_name = os.path.basename(file_path)
        print(f"Mengunggah {file_name}...")
        session.file.put(f"file://{file_path}", data_stage_path, auto_compress=False, overwrite=True)
    else:
        print(f"Skip {file_path} file was not found at /tmp/")

print("\n✓ Automation process Complete! Mari kita cek isi Stage saat ini:")
session.sql(f"LIST {data_stage_path}").show()